In [3]:
pdb_path_80 = "/home/abharadwaj1/papers/elife_paper/figure_information/data/synthetic_anisotropy_spike_protein_renamed/job080/locscale_job_hybrid/conformation_001000_fit_to_job080.pdb"
pdb_path_15 = "/home/abharadwaj1/papers/elife_paper/figure_information/data/synthetic_anisotropy_spike_protein_renamed/job015/locscale_job_hybrid/conformation_001000_fit_to_job015.pdb"

assert os.path.exists(pdb_path_80), f"File not found: {pdb_path_80}"
assert os.path.exists(pdb_path_15), f"File not found: {pdb_path_15}"


In [74]:
import gemmi

def remove_residue(pdb_path: str, residue_name: str, output_path: str = None) -> gemmi.Structure:
    """
    Remove all residues with the given name from a PDB structure.
    
    Args:
        pdb_path: Path to the input PDB file.
        residue_name: Three-letter residue name to remove (e.g. "NM5").
        output_path: Optional path to write the modified structure. If None, no file is written.
    
    Returns:
        The modified gemmi Structure object.
    """
    st = gemmi.read_structure(pdb_path)
    num_atoms = st[0].count_atom_sites()
    print(f"Original structure has {num_atoms} atoms.")
    for model in st:
        for chain in model:
            to_delete = [i for i, res in enumerate(chain) if res.name == residue_name]
            for i in reversed(to_delete):
                del chain[i]
    num_atoms_new = st[0].count_atom_sites()
    print(f"Modified structure has {num_atoms_new} atoms after removing residues named '{residue_name}'.")
    if output_path:
        st.write_pdb(output_path)

    return st

In [7]:
pdb_path_80_modified = pdb_path_80.replace(".pdb", "_modified.pdb")
pdb_path_15_modified = pdb_path_15.replace(".pdb", "_modified.pdb")

remove_residue(pdb_path_80, "NM5", pdb_path_80_modified)
remove_residue(pdb_path_15, "NM5", pdb_path_15_modified)

<gemmi.Structure conformation_001000_fit_to_job015 with 1 model(s)>

In [24]:
st_temp = gemmi.read_structure(pdb_path_80_modified)
print(len(st_temp[0]["A"]))
res = st_temp[0]["A"][0]
res.seqid
# seq_ids_to_check: 1147 1153 1155 1157 1159 1161 1165 1167 1177 1179
seqids_to_check = [1147, 1153, 1155, 1157, 1159, 1161, 1165, 1167, 1177, 1179]
for res in st_temp[0]["A"]:
    if res.seqid.num in seqids_to_check[0:1]:
        print(f"Residue {res.name} at seqid {res.seqid} is present in the modified structure.")
        # check for duplicate atoms 
        atom_position = [] 
        duplicate_atoms_found = 0
        for atom in res:
            pos = (atom.name)
            if pos in atom_position:
                print(f"Duplicate atom found: {atom.name} in residue {res.name} at seqid {res.seqid}")
                duplicate_atoms_found += 1
                
            else:
                atom_position.append(pos)




            

1130
Residue SER at seqid 1147 is present in the modified structure.
Duplicate atom found: N in residue SER at seqid 1147
Duplicate atom found: C in residue SER at seqid 1147
Duplicate atom found: H in residue SER at seqid 1147


In [68]:
def remove_duplicated_atoms(pdb_path, output_pdb_path): 
    st = gemmi.read_structure(pdb_path)
    num_atoms_original = st[0].count_atom_sites()
    num_atoms_removed = 0 
    for model in st:
        for chain in model:
            residue_delete_index = [] 
            for i, res in enumerate(chain):
                #if res.seqid.num in seqids_to_check:
                    #print(f"Residue {res.name} at seqid {res.seqid} is present in the modified structure.")
                atom_positions = []
                duplicate_atoms_found = 0
                for atom in res: 
                    pos = (atom.name)
                    if pos in atom_positions: 
                        #print(f"Duplicate atom found: {atom.name} in residue {res.name} at seqid {res.seqid}")
                        duplicate_atoms_found += 1
                        residue_delete_index.append(i)
                    else:
                        atom_positions.append(pos)
            
            if duplicate_atoms_found > 0: 
                residue_delete_index.append(i)

            
            unique_residue_delete_index = list(set(residue_delete_index))
            unique_residue_delete_index.sort()
            for i in reversed(unique_residue_delete_index):
                del chain[i]
            
    
    print(f"Original number of atoms: {num_atoms_original}")
    num_atoms_new = st[0].count_atom_sites()
    print(f"New number of atoms: {num_atoms_new}")

    st.write_pdb(output_pdb_path)
    
                    



In [70]:
remove_duplicated_atoms(pdb_path_80_modified.replace(".pdb", "_deduplicate.pdb"), pdb_path_80_modified.replace(".pdb", "_deduplicate.pdb"))

Original number of atoms: 51750
New number of atoms: 51750


In [76]:
remove_duplicated_atoms(pdb_path_80_modified, pdb_path_80_modified.replace(".pdb", "_deduplicate.pdb"))
remove_duplicated_atoms(pdb_path_15_modified, pdb_path_15_modified.replace(".pdb", "_deduplicate.pdb"))


Original number of atoms: 57756
New number of atoms: 51750
Original number of atoms: 57756
New number of atoms: 51750


In [72]:
deduplicated_pdb_path_80 = pdb_path_80_modified.replace(".pdb", "_deduplicate.pdb")
deduplicated_pdb_path_15 = pdb_path_15_modified.replace(".pdb", "_deduplicate.pdb")

In [77]:
remove_residue(deduplicated_pdb_path_80, "NLM", deduplicated_pdb_path_80)
remove_residue(deduplicated_pdb_path_15, "NLM", deduplicated_pdb_path_15)

Original structure has 51750 atoms.
Modified structure has 51126 atoms after removing residues named 'NLM'.
Original structure has 51750 atoms.
Modified structure has 51126 atoms after removing residues named 'NLM'.


<gemmi.Structure conformation_001000_fit_to_job015_modified_deduplicate with 1 model(s)>

In [85]:
standard_residues = ["ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL"]
non_standard_residues_80 = []
non_standard_residues_15 = []
st_80 = gemmi.read_structure(deduplicated_pdb_path_15)
for chain in st_80[0]:
    for res in chain:
        if res.name not in standard_residues:
            non_standard_residues_80.append((res.name, res.seqid.num))
            print(f"Non-standard residue in 80% anisotropy structure: {res.name} at seqid {res.seqid}")


In [80]:
deduplicated_pdb_path_80

'/home/abharadwaj1/papers/elife_paper/figure_information/data/synthetic_anisotropy_spike_protein_renamed/job080/locscale_job_hybrid/conformation_001000_fit_to_job080_modified_deduplicate.pdb'

In [86]:
# compute map to model correlation 
from locscale.include.emmer.pdb.fitting_tools import compute_model_to_map_correlation

target_pdb_80 = deduplicated_pdb_path_80
target_pdb_15 = deduplicated_pdb_path_15

map_path_80 = "/home/abharadwaj1/papers/elife_paper/figure_information/data/synthetic_anisotropy_spike_protein_renamed/job080/locscale_job_hybrid/high_anisotropy_locscale.mrc"
map_path_15 = "/home/abharadwaj1/papers/elife_paper/figure_information/data/synthetic_anisotropy_spike_protein_renamed/job015/locscale_job_hybrid/low_anisotropy_locscale.mrc"
input_map_80 = "/home/abharadwaj1/papers/elife_paper/figure_information/data/synthetic_anisotropy_spike_protein_renamed/job080/locscale_job_hybrid/job080_run_class001_zflip.mrc"

correlation_80 = compute_model_to_map_correlation(target_pdb_80, map_path_80)
correlation_15 = compute_model_to_map_correlation(target_pdb_15, map_path_15)
correlation_input_80 = compute_model_to_map_correlation(target_pdb_80, input_map_80)

In [91]:
input_map_15 = "/home/abharadwaj1/papers/elife_paper/figure_information/data/synthetic_anisotropy_spike_protein_renamed/job015/locscale_job_hybrid/job015_run_class001_zflip.mrc"
correlation_input_15 = compute_model_to_map_correlation(target_pdb_15, input_map_15)

In [92]:
print(f"Correlation of 80% anisotropy model to input map: {correlation_input_80}")
print(f"Correlation of 80% anisotropy model to map: {correlation_80}")
print(f"Correlation of 15% anisotropy model to input map: {correlation_input_15}")
print(f"Correlation of 15% anisotropy model to map: {correlation_15}")



Correlation of 80% anisotropy model to input map: 0.3106285735752607
Correlation of 80% anisotropy model to map: 0.3373683940226446
Correlation of 15% anisotropy model to input map: 0.8160440559713651
Correlation of 15% anisotropy model to map: 0.8982907804217097
